# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q dask[complete] pyarrow pandas numpy faker
    print('Packages installed.')

# Handling Large Datasets In-Depth

A common hurdle in social science research is the "Out of Memory" error. If you have a 10GB dataset and only 8GB of RAM, you cannot open it with standard `pd.read_csv()`.

In this notebook, we explore strategies to handle large data:
1.  **Chunking**: Processing data in small pieces.
2.  **Dask**: Parallel computing with lazy evaluation.
3.  **Efficient Formats**: Using Parquet instead of CSV.

In [ ]:
import pandas as pd
import numpy as np
import os
import dask.dataframe as dd
import time

filename = "large_dummy_data.csv"
print("Libraries loaded.")

## 1. Generate a "Large" Dataset

We will generate a CSV file with 1 million rows to simulate a large dataset.

In [ ]:
# Generate 1M rows
n_rows = 1_000_000

df = pd.DataFrame({
    'id': np.arange(n_rows),
    'group': np.random.choice(['A', 'B', 'C', 'D'], size=n_rows),
    'value1': np.random.normal(0, 1, n_rows),
    'value2': np.random.normal(10, 5, n_rows)
})

print(f"Saving {n_rows} rows to {filename}...")
df.to_csv(filename, index=False)
file_size = os.path.getsize(filename) / (1024 * 1024)
print(f"File created. Size: {file_size:.2f} MB")

# Clear memory
del df

## 2. Pandas Chunking

Instead of loading everything at once, we iterate through the file in chunks.

In [ ]:
chunk_size = 100_000
total_sum = 0

start_time = time.time()

for chunk in pd.read_csv(filename, chunksize=chunk_size):
    # Process each chunk (e.g., calculate sum of value1)
    total_sum += chunk['value1'].sum()

end_time = time.time()
print(f"Total Sum: {total_sum:.2f}")
print(f"Time taken (Chunking): {end_time - start_time:.4f} seconds")

## 3. Dask DataFrames

Dask looks and feels like Pandas, but it processes data in parallel and doesn't load everything into RAM.

In [ ]:
start_time = time.time()

# Dask read_csv is lazy
ddf = dd.read_csv(filename)

# Compute mean of value1 by group
result = ddf.groupby('group')['value1'].mean().compute()

end_time = time.time()

print("Group Means:")
print(result)
print(f"Time taken (Dask): {end_time - start_time:.4f} seconds")

## 4. Parquet Format

CSV is slow (text parsing). Parquet is a binary, columnar format that is much faster and smaller.

In [ ]:
# Convert CSV to Parquet
ddf = dd.read_csv(filename)
ddf.to_parquet("data.parquet", engine="pyarrow")

parquet_size = sum(os.path.getsize(os.path.join("data.parquet", f)) for f in os.listdir("data.parquet") if os.path.isfile(os.path.join("data.parquet", f))) / (1024 * 1024)
print(f"Parquet Size: {parquet_size:.2f} MB")

In [ ]:
start_time = time.time()

# Read Parquet
ddf_pq = dd.read_parquet("data.parquet")
result = ddf_pq.groupby('group')['value1'].mean().compute()

end_time = time.time()
print(f"Time taken (Parquet): {end_time - start_time:.4f} seconds")

In [ ]:
# Cleanup
import shutil
if os.path.exists(filename):
    os.remove(filename)
if os.path.exists("data.parquet"):
    shutil.rmtree("data.parquet")
print("Cleanup done.")